# VERITAS OW v4 vs Baseline B7 — Robustness Stress Test (All 10,905 OpenForensics Test Images)

Evaluates the **already-trained saved checkpoints** (`baseline_v4proto_best.pt` and `veritas_v4_v4proto_best.pt`, zero retraining) on **all 10,905 OpenForensics Test images** across 5 conditions:
1. `clean` (original OpenForensics Test + error overlap, Real-mask coupling, and blend checks)
2. `jpeg_q50` (moderate social-media JPEG recompression)
3. `jpeg_q30` (heavy Messenger/WhatsApp JPEG recompression)
4. `blur_sigma2` (Gaussian blur, `k=5, sigma=2.0`)
5. `resize_0.5x` (downscale 50% -> upscale back)


In [ ]:
import json
from pathlib import Path
import torch

WORKING = Path("/kaggle/working/ow_v4_robust") if Path("/kaggle/working").exists() else Path.cwd() / "ow_v4_robust"
WORKING.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "working_dir": str(WORKING),
    "image_size": 600,
    "test_per_class": None,  # ALL 10,905 OpenForensics Test images
    "batch_size_per_gpu": 12,
    "num_workers": 2,
    "conditions": ["clean", "jpeg_q50", "jpeg_q30", "blur_sigma2", "resize_0.5x"],
}

cfg_path = WORKING / "config.json"
cfg_path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"GPUs detected: {n_gpus}", [torch.cuda.get_device_name(i) for i in range(n_gpus)])


In [ ]:
SCRIPT_CODE = 'from __future__ import annotations\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport os\nimport shutil\nimport time\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom scipy.stats import binomtest\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score\nfrom torch.utils.data import DataLoader, Dataset\nfrom torch.utils.data.distributed import DistributedSampler\nfrom torchvision.models import EfficientNet_B7_Weights, efficientnet_b7\n\n# === Embedded VERITASV4 architecture ===\n"""VERITAS OW v4 architecture: zero-sync spatial forensic streams, multi-scale\ndeep-supervised segmentation, and active mask-guided contrastive fusion.\n\nKey architectural fixes over OW v3 (architecture_v3.py):\n1. Zero GPU->CPU syncs in ForensicStreamsV4: replaces per-sample Python loops\n   and `content_bounds()` `.tolist()` calls with fully vectorized GPU ops,\n   content-mean letterbox fill before batched 2D rfft2/irfft2 directional\n   band-pass filtering, and 2-px eroded valid-mask gating on SRM residuals.\n2. Spatial multi-scale SRM & frequency encoders (38x38 and 19x19) instead of\n   tiny 3-layer 1D-flattened CNNs.\n3. Multi-scale ActiveSegmentationHeadV4 taps EfficientNet-B7 stages 2 (150x150),\n   5 (38x38), AND 7 (19x19) plus spatial SRM+Freq maps (38x38), so segmentation\n   supervises deep encoder stages (6-7) and both forensic branches.\n4. Active Mask-Guided Foreground-vs-Background Contrastive Pooling feeds the\n   predicted segmentation probability map M back into classification across RGB,\n   SRM, and Freq streams (global, fg, bg, |fg - bg|, and mask summary stats).\n5. Per-stream independent LayerNorm + balanced 256-d projections + learnable\n   gated fusion + auxiliary forensic-only classification logits during training\n   prevent EfficientNet-B7 from drowning out the forensic branches.\n"""\n\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torchvision.models import EfficientNet_B7_Weights, efficientnet_b7\n\nimport numpy as np\n\nIMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)\nIMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)\n\n\nclass ClassificationHead(nn.Module):\n    def __init__(self, in_ch: int = 2560, dropout: float = 0.2):\n        super().__init__()\n        self.pool = nn.AdaptiveAvgPool2d(1)\n        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_ch, 1))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.fc(self.pool(x).flatten(1))\n\n\nIMPLEMENTATION_VERSION = "ow_v4"\n\n\ndef _srm_kernels_v4() -> torch.Tensor:\n    """Pure-PyTorch 3x1x5x5 SRM high-pass kernels (NumPy-version independent)."""\n    raw = torch.tensor(\n        [\n            [\n                [0, 0, 0, 0, 0],\n                [0, -1, 2, -1, 0],\n                [0, 2, -4, 2, 0],\n                [0, -1, 2, -1, 0],\n                [0, 0, 0, 0, 0],\n            ],\n            [\n                [-1, 2, -2, 2, -1],\n                [2, -6, 8, -6, 2],\n                [-2, 8, -12, 8, -2],\n                [2, -6, 8, -6, 2],\n                [-1, 2, -2, 2, -1],\n            ],\n            [\n                [0, 0, 0, 0, 0],\n                [0, 0, 0, 0, 0],\n                [0, 1, -2, 1, 0],\n                [0, 0, 0, 0, 0],\n                [0, 0, 0, 0, 0],\n            ],\n        ],\n        dtype=torch.float32,\n    )\n    norms = raw.abs().sum(dim=(-2, -1), keepdim=True).clamp_min(1.0)\n    return (raw / norms).unsqueeze(1)\n\n\ndef _gn(channels: int) -> nn.GroupNorm:\n    return nn.GroupNorm(math.gcd(32, channels), channels)\n\n\nclass SeparableConvBlock(nn.Sequential):\n    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, dilation: int = 1):\n        super().__init__(\n            nn.Conv2d(\n                in_ch,\n                in_ch,\n                kernel_size=3,\n                stride=stride,\n                padding=dilation,\n                dilation=dilation,\n                groups=in_ch,\n                bias=False,\n            ),\n            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),\n            _gn(out_ch),\n            nn.SiLU(inplace=True),\n        )\n\n\nclass ResidualForensicBlock(nn.Module):\n    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):\n        super().__init__()\n        self.conv1 = SeparableConvBlock(in_ch, out_ch, stride=stride)\n        self.conv2 = nn.Sequential(\n            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, groups=out_ch, bias=False),\n            nn.Conv2d(out_ch, out_ch, kernel_size=1, bias=False),\n            _gn(out_ch),\n        )\n        self.skip = (\n            nn.Identity()\n            if (stride == 1 and in_ch == out_ch)\n            else nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), _gn(out_ch))\n        )\n        self.act = nn.SiLU(inplace=True)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.act(self.conv2(self.conv1(x)) + self.skip(x))\n\n\nclass ForensicStreamsV4(nn.Module):\n    """Vectorized, zero-sync color SRM + spatial directional FFT band-pass.\n\n    Returns two pixel-aligned 9-channel tensors at input resolution (H, W):\n      - `srm`: 3 SRM high-pass filters x 3 RGB channels, tanh-bounded, gated by\n        an eroded valid_mask so letterbox boundaries never create false spikes.\n      - `freq`: 3 spatial frequency bands (mid radial, high horizontal/vertical,\n        high diagonal) x 3 RGB channels via batched float32 rfft2 -> irfft2,\n        with letterbox pixels replaced by per-image content mean prior to FFT.\n    """\n\n    def __init__(self):\n        super().__init__()\n        self.register_buffer("srm_weight", _srm_kernels_v4())\n        self._band_cache: dict[tuple[int, int, str, int | None], torch.Tensor] = {}\n\n    def _get_freq_masks(self, h: int, w: int, device: torch.device) -> torch.Tensor:\n        key = (h, w, device.type, device.index)\n        cached = self._band_cache.get(key)\n        if cached is not None:\n            return cached\n        fy = torch.fft.fftfreq(h, device=device)[:, None]\n        fx = torch.fft.rfftfreq(w, device=device)[None, :]\n        radius = torch.sqrt(fy.square() + fx.square())\n        abs_fy = fy.abs()\n        abs_fx = fx.abs()\n\n        # Smooth sigmoid-tapered masks to avoid ringing artifacts from hard cutoffs.\n        low_cut = torch.sigmoid((radius - 0.08) * 40.0)\n        mid_hi = torch.sigmoid((radius - 0.25) * 40.0)\n        mid_band = low_cut * (1.0 - mid_hi)\n\n        # Split high frequencies into axis-aligned (JPEG/grid) vs diagonal (blending/GAN).\n        hv_ratio = torch.abs(abs_fx - abs_fy) / (abs_fx + abs_fy + 1e-6)\n        high_hv = mid_hi * hv_ratio\n        high_diag = mid_hi * (1.0 - hv_ratio)\n\n        masks = torch.stack([mid_band, high_hv, high_diag], dim=0)  # [3, H, W//2+1]\n        if len(self._band_cache) > 8:\n            self._band_cache.clear()\n        self._band_cache[key] = masks\n        return masks\n\n    @staticmethod\n    def _erode_valid_mask(valid_mask: torch.Tensor) -> torch.Tensor:\n        # 5x5 erosion removes 2px border where 5x5 SRM kernels straddle padding.\n        return -F.max_pool2d(-valid_mask, kernel_size=5, stride=1, padding=2)\n\n    def forward(\n        self, normalized_rgb: torch.Tensor, valid_mask: torch.Tensor | None = None\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        with torch.autocast(device_type=normalized_rgb.device.type, enabled=False):\n            x = normalized_rgb.float()\n            mean = IMAGENET_MEAN.to(device=x.device, dtype=torch.float32)\n            std = IMAGENET_STD.to(device=x.device, dtype=torch.float32)\n            rgb = (x * std + mean).clamp(0.0, 1.0)\n\n            if valid_mask is None:\n                vm = torch.ones((rgb.shape[0], 1, rgb.shape[-2], rgb.shape[-1]), device=rgb.device, dtype=torch.float32)\n                eroded = vm\n                rgb_filled = rgb\n            else:\n                vm = valid_mask.to(device=rgb.device, dtype=torch.float32)\n                eroded = self._erode_valid_mask(vm)\n                # Fill letterbox padding with per-image content mean before FFT to avoid edge jumps.\n                denom = vm.sum(dim=(-2, -1), keepdim=True).clamp_min(1.0)\n                content_mean = (rgb * vm).sum(dim=(-2, -1), keepdim=True) / denom\n                rgb_filled = rgb * vm + content_mean * (1.0 - vm)\n\n            # 1) Color SRM residuals (9 channels)\n            srm_w = self.srm_weight.float().repeat(3, 1, 1, 1)\n            srm = F.conv2d(rgb_filled, srm_w, padding=2, groups=3).tanh() * eroded\n\n            # 2) Vectorized spatial directional frequency band-pass (9 channels)\n            h, w = rgb_filled.shape[-2:]\n            spec = torch.fft.rfft2(rgb_filled, norm="ortho")  # [N, 3, H, W//2+1]\n            masks = self._get_freq_masks(h, w, rgb_filled.device)  # [3, H, W//2+1]\n            filtered = (spec.unsqueeze(2) * masks.unsqueeze(0).unsqueeze(0)).flatten(1, 2)\n            spatial_freq = torch.fft.irfft2(filtered, s=(h, w), norm="ortho")\n            freq = torch.tanh(spatial_freq * 8.0) * eroded\n\n        return srm, freq\n\n\nclass SpatialForensicEncoderV4(nn.Module):\n    """Multi-scale spatial encoder aligned with EfficientNet-B7 stages 5 & 7/8.\n\n    Produces:\n      - `mid` at ~H/16 (38x38 for 600px, 128 ch) -> feeds ASPP segmentation head\n      - `top` at ~H/32 (19x19 for 600px, 256 ch) -> feeds mask-guided fusion\n    """\n\n    def __init__(self, in_ch: int = 9, mid_ch: int = 128, top_ch: int = 256):\n        super().__init__()\n        self.stem = nn.Sequential(\n            nn.Conv2d(in_ch, 32, kernel_size=3, stride=2, padding=1, bias=False),\n            _gn(32),\n            nn.SiLU(inplace=True),\n        )\n        self.stage1 = ResidualForensicBlock(32, 64, stride=2)   # /4  (150)\n        self.stage2 = ResidualForensicBlock(64, 96, stride=2)   # /8  (75)\n        self.stage3 = nn.Sequential(\n            ResidualForensicBlock(96, mid_ch, stride=2),        # /16 (38)\n            ResidualForensicBlock(mid_ch, mid_ch, stride=1),\n        )\n        self.stage4 = nn.Sequential(\n            ResidualForensicBlock(mid_ch, top_ch, stride=2),    # /32 (19)\n            ResidualForensicBlock(top_ch, top_ch, stride=1),\n        )\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        x = self.stem(x)\n        x = self.stage1(x)\n        x = self.stage2(x)\n        mid = self.stage3(x)\n        top = self.stage4(mid)\n        return mid, top\n\n\nclass MultiScaleSeparableASPP(nn.Module):\n    def __init__(self, in_ch: int, out_ch: int = 256, rates: tuple[int, ...] = (3, 6, 12)):\n        super().__init__()\n        self.branches = nn.ModuleList(\n            [nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), _gn(out_ch), nn.SiLU(inplace=True))]\n        )\n        self.branches.extend(SeparableConvBlock(in_ch, out_ch, dilation=r) for r in rates)\n        self.gap = nn.Sequential(\n            nn.AdaptiveAvgPool2d(1),\n            nn.Conv2d(in_ch, out_ch, 1, bias=False),\n            _gn(out_ch),\n            nn.SiLU(inplace=True),\n        )\n        self.project = nn.Sequential(\n            nn.Conv2d(out_ch * (len(rates) + 2), out_ch, 1, bias=False),\n            _gn(out_ch),\n            nn.SiLU(inplace=True),\n            nn.Dropout2d(0.1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        feats = [b(x) for b in self.branches]\n        feats.append(F.interpolate(self.gap(x), size=x.shape[-2:], mode="bilinear", align_corners=False))\n        return self.project(torch.cat(feats, dim=1))\n\n\nclass ActiveSegmentationHeadV4(nn.Module):\n    """Deeply supervised multi-scale segmentation decoder.\n\n    Fuses EfficientNet-B7 stage 7 (640 ch, 19x19) + stage 5 (224 ch, 38x38) +\n    spatial SRM (128 ch) + spatial Freq (128 ch) before ASPP, then refines with\n    stage 2 (48 ch, 150x150). Both deep RGB semantics and forensic streams get\n    direct pixel-level localization gradients.\n    """\n\n    def __init__(\n        self,\n        high_ch: int = 640,\n        mid_ch: int = 224,\n        low_ch: int = 48,\n        forensic_mid_ch: int = 128,\n        use_forensic: bool = True,\n    ):\n        super().__init__()\n        self.use_forensic = bool(use_forensic)\n        self.high_proj = nn.Sequential(nn.Conv2d(high_ch, 192, 1, bias=False), _gn(192), nn.SiLU(inplace=True))\n        self.mid_proj = nn.Sequential(nn.Conv2d(mid_ch, 128, 1, bias=False), _gn(128), nn.SiLU(inplace=True))\n        fused_ch = 192 + 128\n        if self.use_forensic:\n            self.forensic_proj = nn.Sequential(\n                nn.Conv2d(forensic_mid_ch * 2, 128, 1, bias=False),\n                _gn(128),\n                nn.SiLU(inplace=True),\n            )\n            fused_ch += 128\n        self.aspp = MultiScaleSeparableASPP(fused_ch, out_ch=256, rates=(3, 6, 12))\n        self.low_proj = nn.Sequential(nn.Conv2d(low_ch, 48, 1, bias=False), _gn(48), nn.SiLU(inplace=True))\n        self.decoder = nn.Sequential(\n            SeparableConvBlock(256 + 48, 192),\n            SeparableConvBlock(192, 128),\n            nn.Conv2d(128, 1, kernel_size=1),\n        )\n\n    def forward(\n        self,\n        high: torch.Tensor,\n        mid: torch.Tensor,\n        low: torch.Tensor,\n        out_size: tuple[int, int],\n        srm_mid: torch.Tensor | None = None,\n        freq_mid: torch.Tensor | None = None,\n    ) -> torch.Tensor:\n        target_mid = mid.shape[-2:]\n        parts = [\n            F.interpolate(self.high_proj(high), size=target_mid, mode="bilinear", align_corners=False),\n            self.mid_proj(mid),\n        ]\n        if self.use_forensic and srm_mid is not None and freq_mid is not None:\n            if srm_mid.shape[-2:] != target_mid:\n                srm_mid = F.interpolate(srm_mid, size=target_mid, mode="bilinear", align_corners=False)\n            if freq_mid.shape[-2:] != target_mid:\n                freq_mid = F.interpolate(freq_mid, size=target_mid, mode="bilinear", align_corners=False)\n            parts.append(self.forensic_proj(torch.cat([srm_mid, freq_mid], dim=1)))\n\n        aspp_out = self.aspp(torch.cat(parts, dim=1))\n        up = F.interpolate(aspp_out, size=low.shape[-2:], mode="bilinear", align_corners=False)\n        logits = self.decoder(torch.cat([up, self.low_proj(low)], dim=1))\n        return F.interpolate(logits, size=out_size, mode="bilinear", align_corners=False)\n\n\nclass MaskGuidedContrastiveFusionV4(nn.Module):\n    """Active segmentation-to-classification fusion with balanced per-stream norms.\n\n    Uses the predicted segmentation probability map M to pool each spatial stream\n    (RGB, SRM, Freq) over:\n      - global valid region\n      - predicted tampered foreground (M-weighted)\n      - predicted untouched background ((1 - M)-weighted)\n      - absolute foreground-vs-background discrepancy |fg - bg|\n    Each stream is normalized and projected to the same 256-d width, then mixed\n    by a learned per-image softmax gate so no single stream can drown out others.\n    Also outputs `aux_forensic_logits` (SRM + Freq + mask stats only, no RGB)\n    for auxiliary supervision during training.\n    """\n\n    def __init__(self, rgb_ch: int = 2560, forensic_ch: int = 256, embed_dim: int = 256, dropout: float = 0.25):\n        super().__init__()\n        self.rgb_reduce = nn.Sequential(\n            nn.Conv2d(rgb_ch, embed_dim, kernel_size=1, bias=False),\n            _gn(embed_dim),\n            nn.SiLU(inplace=True),\n        )\n        # Each stream produces [glob, fg, bg, |fg - bg|] -> 4 * C\n        self.rgb_proj = self._stream_mlp(embed_dim * 4, embed_dim, dropout)\n        self.srm_proj = self._stream_mlp(forensic_ch * 4, embed_dim, dropout)\n        self.freq_proj = self._stream_mlp(forensic_ch * 4, embed_dim, dropout)\n        # Dedicated cross-stream foreground-vs-background contrast + mask stats\n        self.contrast_proj = self._stream_mlp(embed_dim + forensic_ch * 2 + 4, embed_dim, dropout)\n\n        # Per-sample soft gating across the 4 balanced 256-d modality tokens\n        self.gate_net = nn.Sequential(\n            nn.LayerNorm(embed_dim * 4),\n            nn.Linear(embed_dim * 4, 128),\n            nn.SiLU(inplace=True),\n            nn.Linear(128, 4),\n        )\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(embed_dim * 4),\n            nn.Dropout(dropout),\n            nn.Linear(embed_dim * 4, embed_dim),\n            nn.SiLU(inplace=True),\n            nn.Dropout(dropout),\n            nn.Linear(embed_dim, 1),\n        )\n        # Auxiliary forensic-only classifier (no RGB) guarantees SRM+Freq stay active\n        self.aux_forensic_cls = nn.Sequential(\n            nn.LayerNorm(embed_dim * 2 + 4),\n            nn.Dropout(dropout),\n            nn.Linear(embed_dim * 2 + 4, 128),\n            nn.SiLU(inplace=True),\n            nn.Linear(128, 1),\n        )\n\n    @staticmethod\n    def _stream_mlp(in_dim: int, out_dim: int, dropout: float) -> nn.Sequential:\n        return nn.Sequential(\n            nn.LayerNorm(in_dim),\n            nn.Dropout(dropout),\n            nn.Linear(in_dim, out_dim),\n            nn.SiLU(inplace=True),\n            nn.LayerNorm(out_dim),\n        )\n\n    @staticmethod\n    def _masked_stats(\n        feat: torch.Tensor, mask_prob: torch.Tensor, valid: torch.Tensor\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        eps = 1e-5\n        w_glob = valid\n        w_fg = mask_prob * valid\n        w_bg = (1.0 - mask_prob) * valid\n\n        glob = (feat * w_glob).sum(dim=(-2, -1)) / w_glob.sum(dim=(-2, -1)).clamp_min(eps)\n        fg = (feat * w_fg).sum(dim=(-2, -1)) / w_fg.sum(dim=(-2, -1)).clamp_min(eps)\n        bg = (feat * w_bg).sum(dim=(-2, -1)) / w_bg.sum(dim=(-2, -1)).clamp_min(eps)\n        fg_frac = w_fg.sum(dim=(-2, -1)) / w_glob.sum(dim=(-2, -1)).clamp_min(eps)\n        bg_frac = w_bg.sum(dim=(-2, -1)) / w_glob.sum(dim=(-2, -1)).clamp_min(eps)\n        contrast_gate = (4.0 * fg_frac * bg_frac).clamp(0.0, 1.0)\n        diff = (fg - bg).abs() * contrast_gate\n        return torch.cat([glob, fg, bg, diff], dim=1), diff\n\n    def forward(\n        self,\n        rgb_top: torch.Tensor,\n        srm_top: torch.Tensor,\n        freq_top: torch.Tensor,\n        seg_prob: torch.Tensor,\n        valid_mask: torch.Tensor | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        grid = rgb_top.shape[-2:]\n        rgb_feat = self.rgb_reduce(rgb_top)\n        if srm_top.shape[-2:] != grid:\n            srm_top = F.interpolate(srm_top, size=grid, mode="bilinear", align_corners=False)\n        if freq_top.shape[-2:] != grid:\n            freq_top = F.interpolate(freq_top, size=grid, mode="bilinear", align_corners=False)\n\n        m = F.interpolate(seg_prob.float(), size=grid, mode="bilinear", align_corners=False).to(rgb_feat.dtype)\n        if valid_mask is None:\n            v = torch.ones_like(m)\n        else:\n            v = F.interpolate(valid_mask.float(), size=grid, mode="area").clamp(0.0, 1.0).to(rgb_feat.dtype)\n            v = (v > 0.1).to(rgb_feat.dtype)\n\n        rgb_vec, rgb_diff = self._masked_stats(rgb_feat, m, v)\n        srm_vec, srm_diff = self._masked_stats(srm_top, m, v)\n        freq_vec, freq_diff = self._masked_stats(freq_top, m, v)\n\n        v_sum = v.sum(dim=(-2, -1)).clamp_min(1e-5)\n        m_mean = (m * v).sum(dim=(-2, -1)) / v_sum\n        m_max = (m * v).amax(dim=(-2, -1))\n        m_var = (((m - m_mean[:, :, None, None]) ** 2) * v).sum(dim=(-2, -1)) / v_sum\n        m_std = torch.sqrt(m_var.clamp_min(1e-6))\n        m_hi = ((m > 0.5).to(m.dtype) * v).sum(dim=(-2, -1)) / v_sum\n        mask_stats = torch.cat([m_mean, m_max, m_std, m_hi], dim=1)  # [N, 4]\n\n        t_rgb = self.rgb_proj(rgb_vec)\n        t_srm = self.srm_proj(srm_vec)\n        t_freq = self.freq_proj(freq_vec)\n        t_con = self.contrast_proj(torch.cat([rgb_diff, srm_diff, freq_diff, mask_stats], dim=1))\n\n        tokens = torch.cat([t_rgb, t_srm, t_freq, t_con], dim=1)\n        gates = torch.softmax(self.gate_net(tokens), dim=1)  # [N, 4]\n        gated_tokens = torch.cat(\n            [\n                t_rgb * (1.0 + gates[:, 0:1]),\n                t_srm * (1.0 + gates[:, 1:2]),\n                t_freq * (1.0 + gates[:, 2:3]),\n                t_con * (1.0 + gates[:, 3:4]),\n            ],\n            dim=1,\n        )\n        cls_logits = self.classifier(gated_tokens)\n        aux_forensic_logits = self.aux_forensic_cls(torch.cat([t_srm, t_freq, mask_stats], dim=1))\n        return cls_logits, aux_forensic_logits, gates\n\n\nclass VERITASV4(nn.Module):\n    """Unified OW v4 model supporting fair head-to-head `baseline` vs `veritas_v4`."""\n\n    def __init__(self, variant: str = "veritas_v4", pretrained: bool = True, dropout: float = 0.25):\n        super().__init__()\n        if variant not in ("baseline", "veritas_v4"):\n            raise ValueError("variant must be \'baseline\' or \'veritas_v4\'")\n        self.variant = variant\n        self.multi_stream = variant == "veritas_v4"\n        self.segmentation = variant == "veritas_v4"\n\n        weights = EfficientNet_B7_Weights.IMAGENET1K_V1 if pretrained else None\n        self.encoder = efficientnet_b7(weights=weights).features\n\n        if variant == "baseline":\n            self.cls_head = ClassificationHead(2560)\n            self.streams = None\n            self.srm_encoder = None\n            self.freq_encoder = None\n            self.seg_head = None\n            self.fusion_head = None\n        else:\n            self.cls_head = None\n            self.streams = ForensicStreamsV4()\n            self.srm_encoder = SpatialForensicEncoderV4(in_ch=9, mid_ch=128, top_ch=256)\n            self.freq_encoder = SpatialForensicEncoderV4(in_ch=9, mid_ch=128, top_ch=256)\n            self.seg_head = ActiveSegmentationHeadV4(\n                high_ch=640, mid_ch=224, low_ch=48, forensic_mid_ch=128, use_forensic=True\n            )\n            self.fusion_head = MaskGuidedContrastiveFusionV4(\n                rgb_ch=2560, forensic_ch=256, embed_dim=256, dropout=dropout\n            )\n\n    def freeze_backbone(self, freeze: bool = True) -> None:\n        for p in self.encoder.parameters():\n            p.requires_grad = not freeze\n\n    def forward(self, x: torch.Tensor, valid_mask: torch.Tensor | None = None) -> dict[str, torch.Tensor]:\n        size = x.shape[-2:]\n        low = mid = high = None\n        feat = x\n        for i, stage in enumerate(self.encoder):\n            feat = stage(feat)\n            if i == 2:\n                low = feat\n            elif i == 5:\n                mid = feat\n            elif i == 7:\n                high = feat\n\n        if self.variant == "baseline":\n            logits = self.cls_head(feat)\n            return {\n                "classification_logits": logits,\n                "fake_probability": logits.sigmoid(),\n            }\n\n        srm_map, freq_map = self.streams(x, valid_mask)\n        srm_mid, srm_top = self.srm_encoder(srm_map)\n        freq_mid, freq_top = self.freq_encoder(freq_map)\n\n        seg_logits = self.seg_head(high, mid, low, out_size=size, srm_mid=srm_mid, freq_mid=freq_mid)\n        seg_prob = seg_logits.sigmoid()\n\n        cls_logits, aux_forensic_logits, stream_gates = self.fusion_head(\n            feat, srm_top, freq_top, seg_prob, valid_mask=valid_mask\n        )\n        return {\n            "classification_logits": cls_logits,\n            "fake_probability": cls_logits.sigmoid(),\n            "aux_forensic_logits": aux_forensic_logits,\n            "segmentation_logits": seg_logits,\n            "segmentation_prob": seg_prob,\n            "stream_gates": stream_gates,\n        }\n\n\nIMAGENET_MEAN_NP = np.array([0.485, 0.456, 0.406], dtype=np.float32)\nIMAGENET_STD_NP = np.array([0.229, 0.224, 0.225], dtype=np.float32)\nEXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}\nLABELS = {"real": 0, "fake": 1}\n\n\ndef setup_ddp():\n    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:\n        rank = int(os.environ["RANK"])\n        world_size = int(os.environ["WORLD_SIZE"])\n        local_rank = int(os.environ.get("LOCAL_RANK", 0))\n        torch.cuda.set_device(local_rank)\n        dist.init_process_group(backend="nccl", init_method="env://")\n        return True, rank, world_size, local_rank, torch.device(f"cuda:{local_rank}")\n    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n    return False, 0, 1, 0, dev\n\n\ndef cleanup_ddp(is_ddp: bool):\n    if is_ddp and dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef discover_wild_root() -> Path:\n    roots = [Path("/kaggle/input")] if Path("/kaggle/input").exists() else []\n    roots.extend([Path.cwd() / "openforensics-wild-dataset", Path.cwd() / "Dataset", Path.cwd()])\n    for root in roots:\n        if not root.exists():\n            continue\n        for dirpath, dirnames, _ in os.walk(root):\n            p = Path(dirpath)\n            if "ow_masks" in p.parts:\n                continue\n            kids = {d.lower() for d in dirnames}\n            if {"train", "validation", "test"} <= kids or {"train", "val", "test"} <= kids:\n                return p\n            if p.name.lower() == "dataset" and ("train" in kids or "test" in kids):\n                return p\n    raise FileNotFoundError("Attach ahmedgamal/openforensics-wild-dataset.")\n\n\ndef _split_dir(root: Path, name: str) -> Path:\n    aliases = {"train": ["Train", "train"], "val": ["Validation", "Val", "validation", "val"], "test": ["Test", "test"]}\n    for a in aliases[name]:\n        p = root / a\n        if p.is_dir():\n            return p\n    raise FileNotFoundError(f"No {name} split under {root}")\n\n\ndef _class_dir(split_dir: Path, name: str):\n    for a in (name, name.capitalize(), name.upper(), name.lower()):\n        p = split_dir / a\n        if p.is_dir():\n            return p\n    return None\n\n\ndef list_images(wild_root: Path, split_name: str) -> pd.DataFrame:\n    split_dir = _split_dir(wild_root, split_name)\n    rows = []\n    for cname, lab in LABELS.items():\n        cdir = _class_dir(split_dir, cname)\n        if cdir is None:\n            continue\n        for p in sorted(cdir.iterdir()):\n            if p.is_file() and p.suffix.lower() in EXTS:\n                rows.append({\n                    "path": str(p),\n                    "split": split_name,\n                    "label": lab,\n                    "class_name": cname,\n                    "file_name": p.name,\n                    "rel_key": f"{split_name}/{cname}/{p.name}",\n                })\n    return pd.DataFrame(rows)\n\n\ndef unpack_from_zips(input_root: Path, cache_dir: Path, wanted_substrings: tuple[str, ...]) -> list[Path]:\n    roots = []\n    if not input_root.exists():\n        return roots\n    for archive in sorted(input_root.rglob("_output_.zip")):\n        key = hashlib.sha256((str(archive) + "|".join(wanted_substrings)).encode()).hexdigest()[:12]\n        target = cache_dir / f"unpacked_{key}"\n        marker = target / ".done"\n        if marker.exists():\n            roots.append(target)\n            continue\n        with zipfile.ZipFile(archive) as z:\n            members = [\n                m for m in z.infolist()\n                if not m.is_dir() and any(s in m.filename.replace("\\\\", "/") for s in wanted_substrings)\n            ]\n            if not members:\n                continue\n            target.mkdir(parents=True, exist_ok=True)\n            for member in members:\n                rel = PurePosixPath(member.filename.replace("\\\\", "/"))\n                if rel.is_absolute() or ".." in rel.parts:\n                    continue\n                dst = target / Path(*rel.parts)\n                dst.parent.mkdir(parents=True, exist_ok=True)\n                with z.open(member) as src, dst.open("wb") as out:\n                    shutil.copyfileobj(src, out)\n            marker.write_text("ok", encoding="utf-8")\n            roots.append(target)\n    return roots\n\n\ndef find_file_in_roots(roots: list[Path], filename: str) -> Path | None:\n    for root in roots:\n        if not root.exists():\n            continue\n        for p in root.rglob(filename):\n            if p.is_file():\n                return p\n    return None\n\n\ndef load_mask_table(working: Path) -> pd.DataFrame:\n    frames = []\n    roots = [p for p in (Path("/kaggle/input"), Path("/kaggle/working"), working, Path.cwd()) if p.exists()]\n    roots.extend(unpack_from_zips(Path("/kaggle/input"), working / "cache", ("ow_masks/",)))\n    seen = set()\n    for root in roots:\n        for dirpath, _, filenames in os.walk(root):\n            if "index.csv" not in filenames:\n                continue\n            path = Path(dirpath) / "index.csv"\n            if str(path) in seen:\n                continue\n            seen.add(str(path))\n            try:\n                df = pd.read_csv(path, dtype=str)\n            except Exception:\n                continue\n            if not {"image_id", "rel_key", "mask_file", "status", "crop_x0"} <= set(df.columns):\n                continue\n            df["_root"] = str(path.parent)\n            frames.append(df)\n    if not frames:\n        raise FileNotFoundError("No ow_masks/index.csv found. Attach clarenceshanley/veritas-ow-00-generate-masks.")\n    all_df = pd.concat(frames, ignore_index=True)\n    kept, used = [], set()\n    for rec in all_df.to_dict("records"):\n        iid = str(rec["image_id"])\n        if iid in used or str(rec.get("status")) != "ok":\n            continue\n        png = Path(str(rec["_root"])) / str(rec["mask_file"])\n        if not png.is_file():\n            continue\n        rec["mask_path"] = str(png)\n        kept.append(rec)\n        used.add(iid)\n    return pd.DataFrame(kept).drop_duplicates("rel_key", keep="last")\n\n\ndef build_test_index(working: Path, rank: int, max_per_class: int | None = None) -> pd.DataFrame:\n    cache_path = working / "cache" / "test_index_robustness.json"\n    if rank == 0:\n        cache_path.parent.mkdir(parents=True, exist_ok=True)\n        wild_root = discover_wild_root()\n        masks = load_mask_table(working)\n        imgs = list_images(wild_root, "test")\n        merged = imgs.merge(\n            masks[["rel_key", "mask_path", "crop_x0", "crop_y0", "crop_x1", "crop_y1", "positive_frac"]],\n            on="rel_key",\n            how="left",\n        )\n        ready = merged.loc[merged["mask_path"].notna()].copy()\n        if max_per_class is not None:\n            parts = []\n            rng = np.random.RandomState(77)\n            for _, g in ready.groupby("label"):\n                n = min(int(max_per_class), len(g))\n                parts.append(g.sample(n=n, random_state=rng) if n < len(g) else g)\n            ready = pd.concat(parts)\n        ready = ready.reset_index(drop=True)\n        print(f"[TEST INDEX] {len(ready)} images (real={(ready.label==0).sum()}, fake={(ready.label==1).sum()})", flush=True)\n        cache_path.write_text(json.dumps(ready.to_dict("records")), encoding="utf-8")\n    if dist.is_initialized():\n        dist.barrier()\n    return pd.DataFrame(json.loads(cache_path.read_text(encoding="utf-8")))\n\n\ndef letterbox_rgb(image_rgb: np.ndarray, size: int) -> tuple[np.ndarray, np.ndarray]:\n    h, w = image_rgb.shape[:2]\n    scale = min(size / max(h, 1), size / max(w, 1))\n    nw, nh = max(int(round(w * scale)), 1), max(int(round(h * scale)), 1)\n    resized = cv2.resize(image_rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)\n    canvas = np.zeros((size, size, 3), dtype=np.uint8)\n    valid = np.zeros((size, size), dtype=np.float32)\n    left, top = (size - nw) // 2, (size - nh) // 2\n    canvas[top : top + nh, left : left + nw] = resized\n    valid[top : top + nh, left : left + nw] = 1.0\n    return canvas, valid\n\n\ndef apply_degradation(crop: np.ndarray, condition: str) -> np.ndarray:\n    if condition == "clean":\n        return crop\n    if condition == "jpeg_q50":\n        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 50])\n        if ok:\n            dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)\n            if dec is not None:\n                return cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)\n        return crop\n    if condition == "jpeg_q30":\n        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 30])\n        if ok:\n            dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)\n            if dec is not None:\n                return cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)\n        return crop\n    if condition == "blur_sigma2":\n        return cv2.GaussianBlur(crop, (5, 5), 2.0)\n    if condition == "resize_0.5x":\n        h, w = crop.shape[:2]\n        small = cv2.resize(crop, (max(16, w // 2), max(16, h // 2)), interpolation=cv2.INTER_AREA)\n        return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)\n    raise ValueError(f"Unknown condition: {condition}")\n\n\nclass StressTestDataset(Dataset):\n    def __init__(self, df: pd.DataFrame, size: int, condition: str):\n        self.df = df.reset_index(drop=True)\n        self.size = int(size)\n        self.condition = condition\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, idx: int):\n        row = self.df.iloc[idx]\n        bgr = cv2.imread(row.path, cv2.IMREAD_COLOR)\n        if bgr is None:\n            raise FileNotFoundError(f"Unreadable image: {row.path}")\n        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)\n        h, w = rgb.shape[:2]\n        x0, y0 = max(0, int(float(row.crop_x0))), max(0, int(float(row.crop_y0)))\n        x1, y1 = min(w, max(x0 + 1, int(float(row.crop_x1)))), min(h, max(y0 + 1, int(float(row.crop_y1))))\n        crop = apply_degradation(rgb[y0:y1, x0:x1], self.condition)\n\n        mask = cv2.imread(str(row.mask_path), cv2.IMREAD_GRAYSCALE)\n        if mask is None:\n            raise FileNotFoundError(f"Unreadable mask: {row.mask_path}")\n        if mask.shape != (self.size, self.size):\n            mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)\n\n        canvas, valid = letterbox_rgb(crop, self.size)\n        x = (canvas.astype(np.float32) / 255.0 - IMAGENET_MEAN_NP) / IMAGENET_STD_NP\n        x = np.transpose(x, (2, 0, 1))\n        target = (mask.astype(np.float32) > 127.0).astype(np.float32)\n        return {\n            "image": torch.from_numpy(x.copy()),\n            "mask": torch.from_numpy(target[None].copy()),\n            "valid_mask": torch.from_numpy(valid[None].copy()),\n            "label": torch.tensor([float(row.label)], dtype=torch.float32),\n            "rel_key": str(row.rel_key),\n        }\n\n\n@torch.inference_mode()\ndef evaluate_condition(model: nn.Module, loader: DataLoader, device: torch.device, is_ddp: bool, is_v4: bool):\n    model.eval()\n    local_recs = []\n    for batch in loader:\n        x = batch["image"].to(device, non_blocking=True).to(memory_format=torch.channels_last)\n        vm = batch["valid_mask"].to(device, non_blocking=True)\n        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):\n            out = model(x, valid_mask=vm)\n        logits = out["classification_logits"].float().squeeze(-1).cpu().numpy()\n        probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -40, 40)))\n        labels = batch["label"].squeeze(-1).numpy().astype(int)\n\n        aux_probs = np.full_like(probs, np.nan)\n        gates_np = np.full((len(probs), 4), np.nan, dtype=np.float32)\n        seg_pos_frac = np.full_like(probs, np.nan)\n        seg_dice = np.full_like(probs, np.nan)\n        seg_iou = np.full_like(probs, np.nan)\n\n        if is_v4:\n            aux_logits = out["aux_forensic_logits"].float().squeeze(-1).cpu().numpy()\n            aux_probs = 1.0 / (1.0 + np.exp(-np.clip(aux_logits, -40, 40)))\n            gates_np = out["stream_gates"].float().cpu().numpy()\n            pred_bin = (out["segmentation_logits"].float().sigmoid() >= 0.5) & (vm > 0.5)\n            gt_bin = (batch["mask"].to(device, non_blocking=True) > 0.5) & (vm > 0.5)\n            vpx = (vm > 0.5).flatten(1).sum(1).clamp_min(1).float()\n            p_sum = pred_bin.flatten(1).sum(1).float()\n            g_sum = gt_bin.flatten(1).sum(1).float()\n            inter = (pred_bin & gt_bin).flatten(1).sum(1).float()\n            union = (pred_bin | gt_bin).flatten(1).sum(1).float()\n            seg_pos_frac = (p_sum / vpx).cpu().numpy()\n            seg_dice = torch.where(g_sum > 0, (2.0 * inter + 1e-6) / (p_sum + g_sum + 1e-6), torch.full_like(inter, float("nan"))).cpu().numpy()\n            seg_iou = torch.where(g_sum > 0, (inter + 1e-6) / (union + 1e-6), torch.full_like(inter, float("nan"))).cpu().numpy()\n\n        for i in range(len(labels)):\n            local_recs.append({\n                "rel_key": batch["rel_key"][i],\n                "label": int(labels[i]),\n                "logit": float(logits[i]),\n                "prob": float(probs[i]),\n                "aux_prob": float(aux_probs[i]),\n                "seg_pos_frac": float(seg_pos_frac[i]),\n                "seg_dice": float(seg_dice[i]),\n                "seg_iou": float(seg_iou[i]),\n                "g_rgb": float(gates_np[i, 0]),\n                "g_srm": float(gates_np[i, 1]),\n                "g_freq": float(gates_np[i, 2]),\n                "g_con": float(gates_np[i, 3]),\n            })\n\n    if is_ddp:\n        gathered = [None] * dist.get_world_size()\n        dist.all_gather_object(gathered, local_recs)\n        all_recs = [r for sub in gathered for r in sub]\n    else:\n        all_recs = local_recs\n\n    dedup = {r["rel_key"]: r for r in all_recs}\n    df = pd.DataFrame(dedup.values()).sort_values("rel_key").reset_index(drop=True)\n    return df\n\n\ndef summarize_pair(b_df: pd.DataFrame, v_df: pd.DataFrame, condition: str) -> dict:\n    m = b_df.merge(v_df, on=["rel_key", "label"], suffixes=("_b", "_v"))\n    y = m["label"].values.astype(int)\n    pb = m["prob_b"].values\n    pv = m["prob_v"].values\n    paux = m["aux_prob_v"].values\n    lb = m["logit_b"].values\n    lv = m["logit_v"].values\n\n    def cls_metrics(p):\n        pred = (p >= 0.5).astype(int)\n        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n        return {\n            "acc": float(accuracy_score(y, pred)),\n            "f1": float(f1_score(y, pred, zero_division=0)),\n            "prec": float(precision_score(y, pred, zero_division=0)),\n            "rec": float(recall_score(y, pred, zero_division=0)),\n            "auc": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else 0.0,\n            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),\n        }\n\n    bm = cls_metrics(pb)\n    vm = cls_metrics(pv)\n    am = cls_metrics(paux)\n    blend_va = cls_metrics(0.5 * pv + 0.5 * paux)\n    blend_ba = cls_metrics(0.5 * pb + 0.5 * paux)\n\n    b_ok = ((pb >= 0.5).astype(int) == y)\n    v_ok = ((pv >= 0.5).astype(int) == y)\n    a_ok = ((paux >= 0.5).astype(int) == y)\n\n    v_only = int((v_ok & ~b_ok).sum())\n    b_only = int((~v_ok & b_ok).sum())\n    p_v = float(binomtest(min(v_only, b_only), v_only + b_only, 0.5).pvalue) if (v_only + b_only) > 0 else 1.0\n\n    a_only = int((a_ok & ~b_ok).sum())\n    b_only_a = int((~a_ok & b_ok).sum())\n    p_a = float(binomtest(min(a_only, b_only_a), a_only + b_only_a, 0.5).pvalue) if (a_only + b_only_a) > 0 else 1.0\n\n    b_err = ~b_ok\n    v_err = ~v_ok\n    shared_err = int((b_err & v_err).sum())\n\n    reals = m[m["label"] == 0]\n    fakes = m[m["label"] == 1]\n    real_corr = float(np.corrcoef(reals["prob_v"], reals["seg_pos_frac_v"])[0, 1]) if len(reals) > 2 and reals["seg_pos_frac_v"].std() > 0 else float("nan")\n\n    return {\n        "condition": condition,\n        "n": int(len(m)),\n        "baseline": bm,\n        "veritas_v4": vm,\n        "aux_forensic_only": am,\n        "blend_v4_aux": blend_va,\n        "blend_b7_aux": blend_ba,\n        "delta_v4_vs_b7_pp": round((vm["acc"] - bm["acc"]) * 100.0, 3),\n        "delta_aux_vs_b7_pp": round((am["acc"] - bm["acc"]) * 100.0, 3),\n        "mcnemar_v4_vs_b7": {"v4_only": v_only, "b7_only": b_only, "p": p_v},\n        "mcnemar_aux_vs_b7": {"aux_only": a_only, "b7_only": b_only_a, "p": p_a},\n        "decision_agreement": float(((pb >= 0.5) == (pv >= 0.5)).mean()),\n        "shared_error_frac_of_v4": float(shared_err / max(1, int(v_err.sum()))),\n        "logit_corr_b7_v4": float(np.corrcoef(lb, lv)[0, 1]) if len(lb) > 2 else float("nan"),\n        "seg_fake_dice": float(np.nanmean(fakes["seg_dice_v"])),\n        "seg_fake_miou": float(np.nanmean(fakes["seg_iou_v"])),\n        "seg_real_pos_frac": float(np.nanmean(reals["seg_pos_frac_v"])),\n        "seg_real_nonempty": float((reals["seg_pos_frac_v"] > 0).mean()),\n        "real_mask_prob_corr": real_corr,\n        "gates": {\n            "rgb": float(np.nanmean(m["g_rgb_v"])),\n            "srm": float(np.nanmean(m["g_srm_v"])),\n            "freq": float(np.nanmean(m["g_freq_v"])),\n            "contrast": float(np.nanmean(m["g_con_v"])),\n        },\n    }\n\n\ndef locate_checkpoints(working: Path, rank: int) -> dict[str, str]:\n    info_path = working / "cache" / "ckpt_paths.json"\n    if rank == 0:\n        info_path.parent.mkdir(parents=True, exist_ok=True)\n        roots = [p for p in (Path("/kaggle/input"), Path("/kaggle/working"), working, Path.cwd()) if p.exists()]\n        b_ckpt = find_file_in_roots(roots, "baseline_v4proto_best.pt")\n        v_ckpt = find_file_in_roots(roots, "veritas_v4_v4proto_best.pt")\n        b_csv = find_file_in_roots(roots, "baseline_v4proto_test_predictions.csv")\n        v_csv = find_file_in_roots(roots, "veritas_v4_v4proto_test_predictions.csv")\n        if b_ckpt is None or v_ckpt is None:\n            extra = unpack_from_zips(\n                Path("/kaggle/input"),\n                working / "cache",\n                ("baseline_v4proto_best.pt", "veritas_v4_v4proto_best.pt", "_test_predictions.csv"),\n            )\n            roots.extend(extra)\n            b_ckpt = b_ckpt or find_file_in_roots(roots, "baseline_v4proto_best.pt")\n            v_ckpt = v_ckpt or find_file_in_roots(roots, "veritas_v4_v4proto_best.pt")\n            b_csv = b_csv or find_file_in_roots(roots, "baseline_v4proto_test_predictions.csv")\n            v_csv = v_csv or find_file_in_roots(roots, "veritas_v4_v4proto_test_predictions.csv")\n        if b_ckpt is None or v_ckpt is None:\n            raise FileNotFoundError(f"Could not find saved checkpoints under /kaggle/input (b={b_ckpt}, v={v_ckpt})")\n        info = {\n            "baseline_ckpt": str(b_ckpt),\n            "veritas_v4_ckpt": str(v_ckpt),\n            "baseline_clean_csv": str(b_csv) if b_csv else "",\n            "veritas_v4_clean_csv": str(v_csv) if v_csv else "",\n        }\n        print("[CHECKPOINTS]", json.dumps(info, indent=2), flush=True)\n        info_path.write_text(json.dumps(info), encoding="utf-8")\n    if dist.is_initialized():\n        dist.barrier()\n    return json.loads(info_path.read_text(encoding="utf-8"))\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    args = parser.parse_args()\n    cfg = json.loads(Path(args.config).read_text(encoding="utf-8"))\n    working = Path(cfg["working_dir"])\n    working.mkdir(parents=True, exist_ok=True)\n\n    is_ddp, rank, world_size, local_rank, device = setup_ddp()\n    try:\n        if rank == 0:\n            print(f"DDP={is_ddp} world_size={world_size} device={device} gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'cpu\'}", flush=True)\n        ckpts = locate_checkpoints(working, rank)\n        test_df = build_test_index(working, rank, max_per_class=cfg.get("test_per_class"))\n\n        # Load models once (pretrained=False because we immediately load saved state_dicts -> zero download delay!)\n        models = {}\n        for variant, ckpt_key in [("baseline", "baseline_ckpt"), ("veritas_v4", "veritas_v4_ckpt")]:\n            m = VERITASV4(variant=variant, pretrained=False).to(device).to(memory_format=torch.channels_last)\n            state = torch.load(ckpts[ckpt_key], map_location=device, weights_only=True)\n            m.load_state_dict(state)\n            m.eval()\n            models[variant] = m\n\n        all_results = []\n        for cond in cfg["conditions"]:\n            t0 = time.time()\n            if rank == 0:\n                print(f"\\n==================== CONDITION: {cond} (n={len(test_df)}) ====================", flush=True)\n            ds = StressTestDataset(test_df, cfg["image_size"], cond)\n            samp = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=False, drop_last=False) if is_ddp else None\n            nw = cfg["num_workers"]\n            loader = DataLoader(ds, batch_size=cfg["batch_size_per_gpu"], sampler=samp, shuffle=False, num_workers=nw, pin_memory=(device.type == "cuda"), persistent_workers=False)\n\n            b_df = evaluate_condition(models["baseline"], loader, device, is_ddp, is_v4=False)\n            v_df = evaluate_condition(models["veritas_v4"], loader, device, is_ddp, is_v4=True)\n\n            if rank == 0:\n                summary = summarize_pair(b_df, v_df, cond)\n                summary["eval_seconds"] = round(time.time() - t0, 1)\n                all_results.append(summary)\n                print(\n                    f"[{cond}] B7={summary[\'baseline\'][\'acc\']*100:.2f}% (AUC {summary[\'baseline\'][\'auc\']:.4f}) | "\n                    f"v4={summary[\'veritas_v4\'][\'acc\']*100:.2f}% (AUC {summary[\'veritas_v4\'][\'auc\']:.4f}, delta {summary[\'delta_v4_vs_b7_pp\']:+.2f} pp, p={summary[\'mcnemar_v4_vs_b7\'][\'p\']:.2e}) | "\n                    f"Aux={summary[\'aux_forensic_only\'][\'acc\']*100:.2f}% (delta {summary[\'delta_aux_vs_b7_pp\']:+.2f} pp, p={summary[\'mcnemar_aux_vs_b7\'][\'p\']:.2e}) | "\n                    f"Dice={summary[\'seg_fake_dice\']:.4f} | r={summary[\'logit_corr_b7_v4\']:.4f} | {summary[\'eval_seconds\']}s",\n                    flush=True,\n                )\n            if is_ddp:\n                dist.barrier()\n\n        if rank == 0:\n            out_json = working / "robustness_summary.json"\n            out_json.write_text(json.dumps({"config": cfg, "results": all_results}, indent=2), encoding="utf-8")\n            print("\\n=== ROBUSTNESS SUMMARY JSON ===", flush=True)\n            print(json.dumps(all_results, indent=2), flush=True)\n    finally:\n        cleanup_ddp(is_ddp)\n\n\nif __name__ == "__main__":\n    main()\n'
script_path = WORKING / 'eval_v4_robustness.py'
script_path.write_text(SCRIPT_CODE, encoding='utf-8')
print('Wrote eval script:', script_path, 'bytes:', len(SCRIPT_CODE))


In [ ]:
import subprocess, sys

n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if n_gpus >= 2:
    cmd = ["torchrun", "--standalone", f"--nproc_per_node={n_gpus}", str(script_path), "--config", str(cfg_path)]
else:
    cmd = [sys.executable, str(script_path), "--config", str(cfg_path)]

print("Launching:", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Robustness eval failed with exit code {rc}")
print("Robustness stress test finished cleanly!")
